For faster downloads and for better GPU, use colab extension on VS Code to connect to a remote Colab Instance

# Checking the Non domain-adapted model

ModernBERT-base is used for its benchmarks and size.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM, Trainer, TrainingArguments, default_data_collator, DataCollatorForLanguageModeling
from datasets import Dataset
import collections
import numpy as np
import math

SEED = 67
CHUNK_SIZE = 128
WWM_PROB = 0.2
BATCH_SIZE = 16 # GPU bound

In [2]:

model_use = "answerdotai/ModernBERT-base"

tokenizer = AutoTokenizer.from_pretrained(model_use)
model = AutoModelForMaskedLM.from_pretrained(model_use)

Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

Attempt on an arbitrary text on a news body

In [7]:
test = " Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week [MASK] as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte\u2019s desk"

In [4]:
inputs = tokenizer(test, return_tensors='pt')
outputs = model(**inputs)
logits = outputs.logits

In [5]:
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]
mask_token_logits = logits[0, mask_token_index, :]
top_10_tokens = torch.topk(mask_token_logits, 10, dim = 1).indices[0].tolist()
for token in top_10_tokens:
    print(f"{tokenizer.decode([token])}, {test.replace(tokenizer.mask_token, tokenizer.decode(token).strip())}")

 stint,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week stint as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte’s desk
 tenure,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week tenure as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte’s desk
 term,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week term as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodrigo Duterte’s desk
 assignment,  Vice President Leni Robredo on Sunday, December 1, said she will publicly release her findings from her two-week assignment as anti-drug body co-chair to make sure that her recommendations will not remain unread on President Rodri

## Preprocessing the Dataset

In [6]:
import json
import pandas as pd

with open('../scraping/rappler_articles.json') as rappler_f:
    rappler_json = json.load(rappler_f)

rappler_df = pd.DataFrame(rappler_json['scraped'])

rappler_df

,title,body
0,Basagan ng Trip with Leloy Claudio: Heroism in...,"MANILA, Philippines – Who is your hero? Can y..."
1,DOH confirms local transmission of COVID-19 De...,The Department of Health (DOH) on Thursday nig...
2,Robin Padilla urges Filipinos to patronize loc...,"MANILA, Philippines – Robin Padilla is known ..."
3,"Pacquiao, Cayetano: Matobato lies ‘damaged’ PH...","MANILA, Philippines – Staunch allies of Presi..."
4,"Bucks erase fourth-quarter deficit, stun Celtics",Bobby Portis turned a missed Giannis Antetokou...
...,...,...
995,A prayer for climate,"Today, September 1, is marked by the Catholic ..."
996,Tim Cone wishes Justin Brownlee well ahead of ...,"MANILA, Philippines – Barangay Ginebra residen..."
997,Olympic hero Farah slams ‘ignorance and prejud...,"LONDON, England – British athletics legend Mo..."
998,UN fears stigmatization of rescued Boko Haram ...,"GENEVA, Switzerland – The United Nations voice..."


In [7]:
def tokenize_func(data):
    tokenized = tokenizer(data["body"])
    if tokenizer.is_fast:
        tokenized["word_ids"] = [tokenized.word_ids(i) for i in range(len(tokenized["input_ids"]))]
    return tokenized


rappler_ds = Dataset.from_pandas(rappler_df)
rappler_ds = rappler_ds.train_test_split(seed=SEED)

tokenized_rappler = rappler_ds.map(
    tokenize_func, batched=True, remove_columns=['title', 'body']
)
tokenized_rappler


Map:   0%|          | 0/750 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'word_ids'],
        num_rows: 750
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'word_ids'],
        num_rows: 250
    })
})

Concatenate all articles and split each article into texts of length chunk size.

In [8]:
def concatenate_chunk(data):
    concatenated = {key: sum(data[key], []) for key in data}
    total_length = len(concatenated[list(data)[0]])
    # remove last when smaller than chunk size
    total_length = (total_length // CHUNK_SIZE) * CHUNK_SIZE
    # split
    splitted = {
        key: [row[i:i + CHUNK_SIZE] for i in range(0, total_length, CHUNK_SIZE)]
        for key, row in concatenated.items()
    }

    splitted["labels"] = splitted["input_ids"].copy()

    return splitted

In [9]:
rappler_chunked = tokenized_rappler.map(concatenate_chunk, batched=True)
rappler_chunked

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 4101
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'word_ids', 'labels'],
        num_rows: 1225
    })
})

In [10]:
tokenizer.decode(rappler_chunked["train"][0]["input_ids"])

'[CLS] PRAIA, Cape Verde – African countries agreed Saturday, February 14, to strengthen their meteorological services to reduce the impact of extreme weather events at a meeting of ministers in Cape Verde. In a declaration adopted at the end of the 5-day African Ministerial Conference on Meteorology (AMCOMET), delegates recognized that “investments in weather and climate services help save lives and property, minimize economic losses and preserve the environment.” Part of the discussion focused on recent natural disasters on the continent, such as deadly floods in January in Malawi and Mozambique. The participants adopted a budget of around one million euros'

## Fine-tuning

In [11]:
# perform whole word masking, not based on tokens, rather on "whole words" based on the collated words gathered during preprocessing
# Adapted from Hugging Face Documentation https://huggingface.co/learn/llm-course/chapter7/3
def whole_word_masking_data_collator(features):
    for feature in features:
        word_ids = feature.pop("word_ids")

        # create a map for a word to list of indices in tokenizer
        mapping = collections.defaultdict(list)
        current_word_index = -1
        current_word = None
        for idx, word_id in enumerate(word_ids):
            if word_id is not None:
                if word_id != current_word:
                    current_word = word_id
                    current_word_index += 1
                mapping[current_word_index].append(idx)
        
        # mask

        mask = np.random.binomial(1, WWM_PROB, (len(mapping), ))
        input_ids = feature["input_ids"]
        labels = feature["labels"]
        new_labels = [-100] * len(labels)
        
        for word_id in np.where(mask)[0]:
            word_id = word_id.item()
            for idx in mapping[word_id]:
                new_labels[idx] = labels[idx]
                input_ids[idx] = tokenizer.mask_token_id
        feature["labels"] = new_labels
    
    return default_data_collator(features)

### Training

In [12]:
logging_steps = len(rappler_chunked["train"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

training_args = TrainingArguments(
    output_dir=f"{model_use.split('/')[-1]}-finetuned-rappler",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_eval_batch_size=BATCH_SIZE,
    per_device_train_batch_size=BATCH_SIZE,
    fp16=True,
    logging_steps=logging_steps
)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=rappler_chunked["train"],
    eval_dataset=rappler_chunked["test"],
    data_collator=data_collator,
)

In [14]:
eval_results = trainer.evaluate()
print(f"Perplexity before fine-tuning: {math.exp(eval_results['eval_loss'])}")

Training Loss,Validation Loss,Epoch
No log,2.042231,0


Perplexity before fine-tuning: 7.7077885784927735


In [15]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,1.662046
2,No log,1.596646
3,No log,1.605551


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=771, training_loss=1.6238049458900778, metrics={'train_runtime': 221.2573, 'train_samples_per_second': 55.605, 'train_steps_per_second': 3.485, 'total_flos': 1048547325100032.0, 'train_loss': 1.6238049458900778, 'epoch': 3.0})

In [16]:
eval_results = trainer.evaluate()
print(f"Perplexity after fine-tuning: {math.exp(eval_results['eval_loss'])}")

Training Loss,Validation Loss,Epoch
No log,1.593953,3


Perplexity after fine-tuning: 4.923170704171452


In [ ]:
trainer.save_model() 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch


tokenizer_test = AutoTokenizer.from_pretrained('./ModernBERT-base-finetuned-rappler')
model_test = AutoModelForMaskedLM.from_pretrained('./ModernBERT-base-finetuned-rappler')


test = "CS Week 2025 [MASK] with a blast as students conquer the halls with glee"
test_input = tokenizer_test(test, return_tensors="pt")

test_output = model_test(**test_input)
test_logits = test_output.logits


mask_token_index = torch.where(test_input["input_ids"] == tokenizer_test.mask_token_id)[1]
mask_token_logits = test_logits[0, mask_token_index, :]
top_10_tokens = torch.topk(mask_token_logits, 10, dim = 1).indices[0].tolist()
for token in top_10_tokens:
    print(f"{tokenizer_test.decode([token])}, {test.replace(tokenizer_test.mask_token, tokenizer_test.decode(token).strip())}")

top_10 = torch.argsort(mask_token_logits, dim = 1, descending=True)
print(top_10)

for t in top_10.tolist():
    print(t)

Loading weights:   0%|          | 0/137 [00:00<?, ?it/s]

 starts, CS Week 2025 starts with a blast as students conquer the halls with glee
 begins, CS Week 2025 begins with a blast as students conquer the halls with glee
 opens, CS Week 2025 opens with a blast as students conquer the halls with glee
 ends, CS Week 2025 ends with a blast as students conquer the halls with glee
 closes, CS Week 2025 closes with a blast as students conquer the halls with glee
 started, CS Week 2025 started with a blast as students conquer the halls with glee
 concludes, CS Week 2025 concludes with a blast as students conquer the halls with glee
 continues, CS Week 2025 continues with a blast as students conquer the halls with glee
 began, CS Week 2025 began with a blast as students conquer the halls with glee
 opened, CS Week 2025 opened with a blast as students conquer the halls with glee
tensor([[ 7866,  9513, 13279,  ..., 19297,  5390, 50284]])
[7866, 9513, 13279, 7637, 27599, 3053, 20097, 7788, 3407, 5485, 745, 7402, 4566, 37048, 30214, 1527, 4581, 16382, 2